**Privacy note:** Notebook outputs were cleared to avoid exposing dataset records. Configure source paths privately using environment variables.

#### Phishing Website Detection

This notebook performs **Exploratory Data Analysis** on a phishing website dataset stored in a SQLite database

The goal is to:
- Load the dataset from SQLite into pandas DataFrame
- Perform data inspection
- Check data quality issues
- Explore distributions and relationships
- Compare feature distributions across target classes (`label`)

#### Connect to SQLite Database

We connect to the SQLite database and check available tables. 

In [ ]:
import sqlite3
import os
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Supply an existing SQLite database containing the "phishing_data" table.
# The original source dataset is not checked into the public repository.
database_path = Path(os.environ["PHISHING_DB_PATH"]).expanduser()
if not database_path.is_file():
    raise FileNotFoundError(
        f"SQLite database not found: {database_path}. "
        "Set PHISHING_DB_PATH to your existing source database."
    )
conn = sqlite3.connect(str(database_path))
cursor = conn.cursor()
cursor.execute("SELECT name FROM sqlite_master WHERE type='table';")
cursor.fetchall()

df = pd.read_sql_query("SELECT * FROM phishing_data", conn)
df = df.drop(columns="Unnamed: 0", errors="ignore")
df.head()

#### Dataset Overview

We examine:
- Shape of the dataset
- Column names
- Data types

In [ ]:
df.shape

In [ ]:
df.describe(include='all')

In [ ]:
df.info()

#### Missing value analysis

We examine missing values across all features to identify incomplete data that may affect model performance or require imputation.

In [ ]:
df.isnull().sum()

#### Feature Exploration

This section explores the dataset's numeric and categorical features. We analyze distributions, detect outliers, and understand overall feature behaviour.

In [ ]:
num_col = df.select_dtypes(include=np.number).columns.tolist()
num_col

In [ ]:
plt.figure(figsize=(5,3))
sns.boxplot(data=df[num_col], orient='h')
plt.title('Boxplot of Features')
plt.show()

In [ ]:
cat_col = df.select_dtypes(include='object').columns.tolist()
cat_col

In [ ]:
for col in cat_col:
    plt.figure(figsize=(5,3))
    df[col].value_counts().plot(kind='bar')
    plt.title(f"Value Counts for {col}")
    plt.show()

#### Target Variable

The target column 'label' is binary: 
- '0' indicates a phishing website
- '1' indicates a legitimate website

We examine class balance to determine whether the dataset is imbalanced

In [ ]:
df['label'].value_counts().plot(kind='bar')
plt.title("Class Distribution (0 = Phishing, 1 = Legit)")
plt.show()

df['label'].value_counts(normalize=True)

#### Correlation Analysis

The correlation heatmap highlights relationships between numerical features and helps identify signals that may differentiate phishing from legitimate websites. 

Notable observations include strong correlations between: 
- 'NoofSelfRedirect' and 'NoofExternalRef', suggesting that pages with frequent redirects also tend to reference many external resources - a common trait in phishing pages
- 'LargestLineLength' and 'LineofCode', which may reflect obfuscated or auto-generated HTML often used to hide malicious behaviour.

While correlation does not imply causation, these relationships indicate feature groups that may jointly contribute to phishing detection.

In [ ]:
plt.figure(figsize=(6,6))
sns.heatmap(df[num_col].corr(), annot=False, cmap='coolwarm')
plt.title("Correlation Heatmap")
plt.show()

#### Feature Distribution by Website Type 

Boxplots are used to compare feature distributions between phishing and legitimate websites.

Several features exhibit visibly different distributions across classes:
- Phishing websites tend to show higher variability and more extreme outliers in features such as 'NoOfExternalRef' and 'LargestLineLength', which may indicate injected scripts or abnormal page structures.
- Legitimate websites generally dsiplay tigher value ranges, reflecting more consistent and well-maintained HTML structures. 

These differences suggest that structural complexity and abnormal link patterns are useful signals when distinguishing phishing websites from legitimate ones. 

In [ ]:
for col in num_col:
    plt.figure(figsize=(3,2))
    sns.boxplot(x=df['label'], y=df[col])
    plt.show()

#### Project Summary

In this notebook, we performed exploratory data analysis (EDA) on a phishing website dataset stored in a SQLite database.

Key steps included:
- Inspecting dataset structure, data types, and summary statistics
- Identifying missing values (notably in `LineOfCode`)
- Exploring categorical and numeric feature distributions
- Detecting outliers using boxplots
- Analyzing feature correlations
- Comparing feature distributions between phishing and legitimate websites

The analysis reveals structural and behavioral differences between phishing and legitimate websites, particularly in features related to redirects, external references, and page structure. 

This EDA provides a strong foundation for future preprocessing and machine learning modeling. 